# Homework: LoRA from Scratch on GPT-2

In this assignment you will:

1. Implement **LoRA** (Low-Rank Adaptation) from scratch as a thin wrapper around `nn.Linear`.
2. Inject it into a frozen pre-trained **GPT-2 (small, 124M)** and verify only the LoRA params are trainable.
3. Fine-tune on **TinyShakespeare** so the model starts producing pseudo-Elizabethan text.
4. Save / load just the adapter (a few MB instead of 500MB).
5. Re-do the same fine-tune with the `peft` library and confirm the results match.
6. **Bonus:** swap the dataset for Rick & Morty dialogue, Hermione Granger lines, or Yoda quotes and watch the style transfer.

The whole thing fits on a free Colab T4 in well under an hour.

> **Submission:** run all cells, keep the printed sample outputs, and save the notebook. Written answers are filled in the question markdown cells.


## 0 · LoRA in one minute

For any frozen linear layer $W_0 \in \mathbb{R}^{d_{\text{out}} \times d_{\text{in}}}$, LoRA adds a learned low-rank update:

$$
h = W_0 x + \Delta W \cdot x, \qquad \Delta W = \frac{\alpha}{r}\, B A
$$

where $A \in \mathbb{R}^{r \times d_{\text{in}}}$ and $B \in \mathbb{R}^{d_{\text{out}} \times r}$ with $r \ll \min(d_{\text{in}}, d_{\text{out}})$.

Key facts:

- $A$ is initialized with Kaiming uniform, $B$ with **zeros** ⇒ $\Delta W = 0$ at init, so the wrapped model behaves identically to the original on step 0.
- Only $A$ and $B$ are trained ⇒ the trainable param count drops by ~3 orders of magnitude.
- At inference you can either (a) keep $A, B$ separate, or (b) merge: $W \leftarrow W_0 + \frac{\alpha}{r} B A$ — same FLOPs as the original.
- $\alpha$ is a scaling hyperparameter; effective learning rate of the update scales with $\alpha/r$.

Reference: Hu et al., *LoRA: Low-Rank Adaptation of Large Language Models*, 2021 — https://arxiv.org/abs/2106.09685


## 1 · Setup

If you're on Colab, run the install cell. Local installs may already have these.


In [1]:
# Colab install. Skip if you already have these locally.
!pip -q install "transformers>=4.40" "datasets>=2.18" "peft>=0.10" "accelerate>=0.27"


In [2]:
import math
import os
import random
from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

import transformers
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

from device import choose_device

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = choose_device()
print("device:", device, "| transformers:", transformers.__version__)


/Users/akhabalov-da_1/Documents/STUDY/nebius-ai-performance-engineering/code/lm-arch-ht4/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: mps | transformers: 5.8.0


## 2 · Inspect GPT-2

We'll use **gpt2** (124M params). Let's load it and see what kinds of layers live inside.

> **Heads up:** HuggingFace's GPT-2 uses `transformers.pytorch_utils.Conv1D` (a quirky historical choice — it's just a linear layer with weight transposed) for `c_attn` and `c_proj`, **not** `nn.Linear`. That makes writing a generic LoRA wrapper annoying. We'll convert these to plain `nn.Linear` first so your LoRA code stays clean.


In [3]:
MODEL_NAME = "gpt2"

tokenizer = GPT2TokenizerFast.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

n_total = sum(p.numel() for p in model.parameters())
print(f"Total params: {n_total/1e6:.1f}M")

# Print one transformer block to see the layer types
print(model.transformer.h[0])


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 10366.34it/s]


Total params: 124.4M
GPT2Block(
  (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (attn): GPT2Attention(
    (c_attn): Conv1D(nf=2304, nx=768)
    (c_proj): Conv1D(nf=768, nx=768)
    (attn_dropout): Dropout(p=0.1, inplace=False)
    (resid_dropout): Dropout(p=0.1, inplace=False)
  )
  (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (mlp): GPT2MLP(
    (c_fc): Conv1D(nf=3072, nx=768)
    (c_proj): Conv1D(nf=768, nx=3072)
    (act): NewGELUActivation()
    (dropout): Dropout(p=0.1, inplace=False)
  )
)


In [4]:
# Helper: convert all transformers Conv1D modules to nn.Linear (functionally identical).
# This is given to you. Read it — but you don't need to modify it.
from transformers.pytorch_utils import Conv1D


def conv1d_to_linear(conv: Conv1D) -> nn.Linear:
    # Conv1D stores weight as (in_features, out_features); nn.Linear stores (out_features, in_features).
    in_features, out_features = conv.weight.shape
    linear = nn.Linear(in_features, out_features, bias=conv.bias is not None)
    with torch.no_grad():
        linear.weight.copy_(conv.weight.T)
        if conv.bias is not None:
            linear.bias.copy_(conv.bias)
    return linear


def replace_conv1d_with_linear(module: nn.Module) -> None:
    for name, child in list(module.named_children()):
        if isinstance(child, Conv1D):
            setattr(module, name, conv1d_to_linear(child).to(next(module.parameters()).device))
        else:
            replace_conv1d_with_linear(child)


replace_conv1d_with_linear(model)

# Sanity-check: the model still produces the same logits as before within fp tolerance.
with torch.no_grad():
    ids = tokenizer("The quick brown fox", return_tensors="pt").input_ids.to(device)
    logits = model(ids).logits
print("After conversion, logits shape:", tuple(logits.shape))
print(model.transformer.h[0].attn)  # c_attn / c_proj are now nn.Linear


After conversion, logits shape: (1, 4, 50257)
GPT2Attention(
  (c_attn): Linear(in_features=768, out_features=2304, bias=True)
  (c_proj): Linear(in_features=768, out_features=768, bias=True)
  (attn_dropout): Dropout(p=0.1, inplace=False)
  (resid_dropout): Dropout(p=0.1, inplace=False)
)


## 3 · Baseline generation (before fine-tuning)

So we have a "before" reference for the style transfer. Save these outputs — you'll diff them against the post-LoRA samples later.


In [5]:
@torch.no_grad()
def generate(prompt: str, max_new_tokens: int = 80, temperature: float = 0.9, top_p: float = 0.95, seed: int = 0) -> str:
    torch.manual_seed(seed)
    model.eval()
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    out = model.generate(
        ids,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)


PROMPTS = [
    "ROMEO:",
    "To be, or not to be,",
    "Once upon a time in fair Verona,",
]

print("=== BASELINE (no fine-tuning) ===")
for p in PROMPTS:
    print("-" * 60)
    print(generate(p, seed=SEED))


[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


=== BASELINE (no fine-tuning) ===
------------------------------------------------------------
ROMEO: I did something I thought was stupid. It did not lead to anything significant. I think I tried to be fair and fair to the players on both sides of the coin."

That may be to do with the fact that he's still a coach with a very long and very long history.

What he really should have done is look at his game during his time as general manager and
------------------------------------------------------------
To be, or not to be, the subject of this, there is a kind of anachronistic, a kind of anachronism, and the fact that we have two different kinds of a moral being who are neither of such a sort nor such anachronistic is a sort of a sort of a sort of a sort of anachronistic. That is one of the ways in which moral thinking has come about
------------------------------------------------------------
Once upon a time in fair Verona, the village of Hira, with its rich population, the city was

## 4 · TODO: Implement `LoRALinear` (10 points)

Wrap an existing `nn.Linear` so the forward pass becomes:

$$
y = W_0 x + b + \frac{\alpha}{r} B A \cdot \text{dropout}(x)
$$

Constraints:

- The base layer's weight and bias must be **frozen** (`requires_grad = False`).
- $A$: shape `(r, in_features)`, initialised with `nn.init.kaiming_uniform_(a=math.sqrt(5))`.
- $B$: shape `(out_features, r)`, initialised with **zeros**. (Why zeros? See the background section.)
- Optional dropout on the input *before* it hits the LoRA branch.
- `merged_weight()` returns $W_0 + \frac{\alpha}{r} B A$ — useful for checking equivalence and for inference-time merging.

Fill in the `# TODO` lines.


In [6]:
class LoRALinear(nn.Module):
    """Frozen linear layer + trainable low-rank residual."""

    def __init__(self, base: nn.Linear, r: int = 8, alpha: int = 16, dropout: float = 0.0):
        super().__init__()
        assert isinstance(base, nn.Linear), "LoRALinear only wraps nn.Linear in this homework."
        self.base = base
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        in_features = base.in_features
        out_features = base.out_features

        # Freeze the original projection. Only the low-rank adapter learns.
        self.base.weight.requires_grad = False
        if self.base.bias is not None:
            self.base.bias.requires_grad = False

        # Keep adapter parameters on the same device/dtype as the wrapped layer.
        self.lora_A = nn.Parameter(
            torch.empty(r, in_features, device=base.weight.device, dtype=base.weight.dtype)
        )
        self.lora_B = nn.Parameter(
            torch.zeros(out_features, r, device=base.weight.device, dtype=base.weight.dtype)
        )
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

        self.lora_dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        lora_update = self.lora_dropout(x) @ self.lora_A.T @ self.lora_B.T
        return self.base(x) + self.scaling * lora_update

    @torch.no_grad()
    def merged_weight(self) -> torch.Tensor:
        return self.base.weight + self.scaling * (self.lora_B @ self.lora_A)



In [7]:
# Sanity check: at init, LoRALinear must be functionally identical to the base layer.
torch.manual_seed(0)
base = nn.Linear(64, 32).to(device)
lora = LoRALinear(base, r=4, alpha=8)  # NOTE: no trailing .to(device) -- LoRALinear should already place params on base's device.

assert lora.lora_A.device == base.weight.device, (
    f"lora_A on {lora.lora_A.device} but base on {base.weight.device}. "
    "Allocate lora_A/lora_B on base.weight.device inside __init__."
)
assert lora.lora_B.device == base.weight.device, "lora_B on wrong device, see hint in TODO 4.2."

x = torch.randn(2, 5, 64, device=device)
y_base = base(x)
y_lora = lora(x)

assert torch.allclose(y_base, y_lora, atol=1e-6), "LoRA forward differs from base at init — B should be zeros!"
print("OK: LoRALinear matches base at init.")

# After perturbing B, outputs should differ.
with torch.no_grad():
    lora.lora_B.add_(torch.randn_like(lora.lora_B))
assert not torch.allclose(base(x), lora(x), atol=1e-6)
print("OK: LoRALinear differs from base after perturbing B.")

# Merged weight equivalence.
with torch.no_grad():
    merged = lora.merged_weight()
    y_merged = F.linear(x, merged, base.bias)
    assert torch.allclose(y_merged, lora(x), atol=1e-5), "merged_weight() inconsistent with forward()."
print("OK: merged_weight() matches forward().")


OK: LoRALinear matches base at init.
OK: LoRALinear differs from base after perturbing B.
OK: merged_weight() matches forward().


## 5 · TODO: Inject LoRA into GPT-2 and freeze the rest (5 points)

You'll write `inject_lora` that walks the model, finds every `nn.Linear` whose attribute name matches `target_names`, and replaces it with a `LoRALinear` wrapping it.

We'll target `("c_attn", "c_proj")`:

- `c_attn` — the fused QKV projection (one per block, in `attn`).
- `c_proj` — *appears in two places* per block: the attention output projection (`attn.c_proj`) **and** the MLP down-projection (`mlp.c_proj`). Matching by attribute name will hit both — that's fine and is consistent with what `peft` does by default (it suffix-matches the same names). It's a useful gotcha to be aware of.

So you should end up with **36 wrappings** = 12 blocks × (1 `c_attn` + 1 `attn.c_proj` + 1 `mlp.c_proj`).

After injection, **only `lora_A` and `lora_B` should have `requires_grad=True`.**


In [8]:
def inject_lora(
    module: nn.Module,
    target_names: tuple[str, ...] = ("c_attn", "c_proj"),
    r: int = 8,
    alpha: int = 16,
    dropout: float = 0.05,
) -> int:
    """Recursively swap nn.Linear children whose attribute name is in target_names.

    Returns the number of modules wrapped.
    """
    n_wrapped = 0
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and name in target_names:
            setattr(module, name, LoRALinear(child, r=r, alpha=alpha, dropout=dropout))
            n_wrapped += 1
        else:
            n_wrapped += inject_lora(child, target_names=target_names, r=r, alpha=alpha, dropout=dropout)
    return n_wrapped


def freeze_non_lora(model: nn.Module) -> None:
    for name, param in model.named_parameters():
        param.requires_grad = "lora_" in name


def count_params(model: nn.Module) -> tuple[int, int]:
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total


def assert_same_device(model: nn.Module) -> None:
    """Sanity check: every parameter lives on the same device. Catches a common LoRA bug."""
    expected = next(model.parameters()).device
    for n, p in model.named_parameters():
        assert p.device == expected, f"param {n} on {p.device}, expected {expected}"



In [9]:
# Reload a fresh GPT-2 (we'll keep the converted Linear version so LoRALinear can wrap it).
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)
replace_conv1d_with_linear(model)

n_wrapped = inject_lora(model, target_names=("c_attn", "c_proj"), r=8, alpha=16, dropout=0.05)
freeze_non_lora(model)

trainable, total = count_params(model)
print(f"Wrapped {n_wrapped} layers")
print(f"Trainable: {trainable/1e6:.3f}M  /  Total: {total/1e6:.1f}M  ({100*trainable/total:.2f}%)")
assert n_wrapped == 36, (
    f"Expected 36 (12 blocks x (c_attn + attn.c_proj + mlp.c_proj)), got {n_wrapped}"
)
assert trainable < 1_500_000, "Way too many trainable params — check freeze_non_lora."
assert_same_device(model)


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4591.60it/s]


Wrapped 36 layers
Trainable: 0.811M  /  Total: 125.3M  (0.65%)


**Q1. (5 points)** What fraction of the original 124M parameters did you make trainable? With $r=8$ and target modules `(c_attn, c_proj)`, you should be around 0.6–0.7%. Think about why: how many params does each LoRA pair add for a `(d_in, d_out)` linear, and what are the dims of the three target layer types?

**MY ANSWER**

## Per-pair formula
- For a `(d_in, d_out)` linear, a single LoRA pair (A of shape `(r, d_in)`, B of shape `(d_out, r)`) adds `r * d_in + d_out * r = r * (d_in + d_out)` parameters.

## Per-layer LoRA params at r=8
- `attn.c_attn`  (768 → 2304): `8 * (768 + 2304) = 24,576` params
- `attn.c_proj`  (768 →  768): `8 * (768 + 768) = 12,288` params
- `mlp.c_proj`   (3072 → 768): `8 * (3072 + 768) = 30,720` params

## Totals
- Trainable parameters across all 12 blocks: `(24,576 + 12,288 + 30,720) * 12 = 811,008`
- Fraction of GPT-2 small (124M): `811,008 / 124,000,000 * 100 ≈ 0.65%`


## 6 · Dataset: TinyShakespeare

A single ~1MB text file. We tokenize the whole thing once, then chunk into fixed-length blocks.


In [10]:
import urllib.request

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)
TINY_SHAKES_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
TINY_SHAKES_PATH = os.path.join(DATA_DIR, "tinyshakespeare.txt")
if not os.path.exists(TINY_SHAKES_PATH):
    urllib.request.urlretrieve(TINY_SHAKES_URL, TINY_SHAKES_PATH)

with open(TINY_SHAKES_PATH) as f:
    raw_text = f.read()

print(f"Corpus: {len(raw_text):,} chars")
print(raw_text[:300])


Corpus: 1,115,394 chars
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


In [11]:
BLOCK_SIZE = 256


class LMChunks(Dataset):
    """Tokenize a long string once, then yield contiguous fixed-length chunks for next-token LM training."""

    def __init__(self, text: str, tokenizer, block_size: int):
        self.block_size = block_size
        ids = tokenizer(text, return_tensors="pt").input_ids[0]
        # Drop the tail so we have a clean multiple of block_size.
        n_chunks = ids.size(0) // block_size
        self.ids = ids[: n_chunks * block_size].view(n_chunks, block_size)

    def __len__(self) -> int:
        return self.ids.size(0)

    def __getitem__(self, idx: int) -> dict:
        chunk = self.ids[idx]
        # For causal LM, labels = input_ids; HF will internally shift by one.
        return {"input_ids": chunk, "labels": chunk.clone()}


full_ds = LMChunks(raw_text, tokenizer, BLOCK_SIZE)
n_val = max(1, len(full_ds) // 20)
train_ds, val_ds = torch.utils.data.random_split(
    full_ds, [len(full_ds) - n_val, n_val], generator=torch.Generator().manual_seed(SEED)
)
print(f"Train chunks: {len(train_ds)} | Val chunks: {len(val_ds)} | block_size={BLOCK_SIZE}")


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (338025 > 1024). Running this sequence through the model will result in indexing errors


Train chunks: 1254 | Val chunks: 66 | block_size=256


## 6.5 · Evaluation harness: perplexity

We need a quantitative sanity check, not just vibes from generated samples. Two metrics:

1. **Shakespeare-val PPL** — should drop sharply after fine-tuning (the model is getting better at predicting Shakespeare tokens).
2. **General-English PPL** — computed on a held-out non-Shakespeare snippet (Pride & Prejudice). This catches **catastrophic forgetting**: if it skyrockets, your LoRA has destroyed general English ability in chasing Shakespeare style. Mild rises are normal; large rises mean your `r`/`alpha`/lr is too aggressive or you're training too long.

> Note: right now `model` already has LoRA injected with $B = 0$, so its outputs are mathematically identical to plain GPT-2. The "baseline" PPL we print below is therefore the un-tuned GPT-2's PPL on each corpus.


In [12]:
def collate(batch):
    return {k: torch.stack([b[k] for b in batch], dim=0) for k in batch[0]}


@torch.no_grad()
def compute_ppl(model: nn.Module, dataset, batch_size: int = 8) -> float:
    """Token-weighted perplexity over a chunk dataset. Lower = better."""
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate)
    total_loss = 0.0
    total_tokens = 0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        # HF computes mean cross-entropy over (B*T) shifted-label positions; weight by that token count.
        n_tok = batch["labels"].numel()
        total_loss += out.loss.item() * n_tok
        total_tokens += n_tok
    return math.exp(total_loss / total_tokens)


In [13]:
# Forgetting-control corpus: a public-domain Pride & Prejudice excerpt (no Shakespeare in sight).
control_text = """It is a truth universally acknowledged, that a single man in possession of a good fortune, must be in want of a wife. However little known the feelings or views of such a man may be on his first entering a neighbourhood, this truth is so well fixed in the minds of the surrounding families, that he is considered as the rightful property of some one or other of their daughters.

"My dear Mr. Bennet," said his lady to him one day, "have you heard that Netherfield Park is let at last?"

Mr. Bennet replied that he had not.

"But it is," returned she; "for Mrs. Long has just been here, and she told me all about it."

Mr. Bennet made no answer.

"Do not you want to know who has taken it?" cried his wife impatiently.

"You want to tell me, and I have no objection to hearing it."

This was invitation enough.

"Why, my dear, you must know, Mrs. Long says that Netherfield is taken by a young man of large fortune from the north of England; that he came down on Monday in a chaise and four to see the place, and was so much delighted with it that he agreed with Mr. Morris immediately; that he is to take possession before Michaelmas, and some of his servants are to be in the house by the end of next week."

"What is his name?"

"Bingley."

"Is he married or single?"

"Oh! single, my dear, to be sure! A single man of large fortune; four or five thousand a year. What a fine thing for our girls!"

"How so? how can it affect them?"

"My dear Mr. Bennet," replied his wife, "how can you be so tiresome! You must know that I am thinking of his marrying one of them."

"Is that his design in settling here?"

"Design! nonsense, how can you talk so! But it is very likely that he may fall in love with one of them, and therefore you must visit him as soon as he comes."
"""

control_ds = LMChunks(control_text, tokenizer, BLOCK_SIZE)
print(f"Control chunks: {len(control_ds)} (block_size={BLOCK_SIZE})")
assert len(control_ds) >= 1, "Control corpus too short — add more text."


Control chunks: 1 (block_size=256)


In [14]:
ppl_shakes_before = compute_ppl(model, val_ds)
ppl_control_before = compute_ppl(model, control_ds)
print(f"Baseline Shakespeare-val PPL : {ppl_shakes_before:7.2f}")
print(f"Baseline Control      PPL    : {ppl_control_before:7.2f}")


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Baseline Shakespeare-val PPL :   93.53
Baseline Control      PPL    :   16.92


## 7 · Training loop

The loop itself is given. **TODO 7.1**: build the optimizer over only the trainable parameters.


In [15]:
@dataclass
class TrainCfg:
    epochs: int = 2
    batch_size: int = 8
    lr: float = 3e-4
    weight_decay: float = 0.0
    warmup_ratio: float = 0.05
    log_every: int = 50
    grad_clip: float = 1.0


def train_lora(model: nn.Module, train_ds, val_ds, cfg: TrainCfg) -> dict:
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate)

    # Build optimizer state only for trainable adapter parameters.
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    assert trainable_params, "No trainable parameters found. Did you inject/freeze LoRA correctly?"
    optim = torch.optim.AdamW(trainable_params, lr=cfg.lr, weight_decay=cfg.weight_decay)

    total_steps = len(train_loader) * cfg.epochs
    warmup_steps = int(cfg.warmup_ratio * total_steps)

    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.0, 0.5 * (1 + math.cos(math.pi * progress)))

    sched = torch.optim.lr_scheduler.LambdaLR(optim, lr_lambda)

    history = {"train_loss": [], "val_loss": []}
    step = 0
    for epoch in range(cfg.epochs):
        model.train()
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            loss = out.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg.grad_clip)
            optim.step()
            sched.step()
            optim.zero_grad(set_to_none=True)

            if step % cfg.log_every == 0:
                history["train_loss"].append((step, loss.item()))
                print(f"epoch {epoch} step {step:4d} | train loss {loss.item():.4f} | lr {sched.get_last_lr()[0]:.2e}")
            step += 1

        # validation
        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                val_losses.append(model(**batch).loss.item())
        v = sum(val_losses) / len(val_losses)
        history["val_loss"].append((step, v))
        print(f"== epoch {epoch} val loss {v:.4f} (ppl {math.exp(v):.2f}) ==")

    return history


cfg = TrainCfg(epochs=2, batch_size=8, lr=3e-4)
history = train_lora(model, train_ds, val_ds, cfg)


epoch 0 step    0 | train loss 4.3586 | lr 2.00e-05
epoch 0 step   50 | train loss 4.2260 | lr 2.89e-04
epoch 0 step  100 | train loss 3.8874 | lr 2.43e-04
epoch 0 step  150 | train loss 3.5215 | lr 1.71e-04
== epoch 0 val loss 3.7247 (ppl 41.46) ==
epoch 1 step  200 | train loss 3.9274 | lr 9.39e-05
epoch 1 step  250 | train loss 3.6619 | lr 3.17e-05
epoch 1 step  300 | train loss 4.0091 | lr 1.40e-06
== epoch 1 val loss 3.7024 (ppl 40.55) ==


## 8 · Generate after fine-tuning

Same prompts, same seed. Read the outputs and compare to the baseline.


In [16]:
print("=== AFTER LoRA FINE-TUNE ===")
for p in PROMPTS:
    print("-" * 60)
    print(generate(p, seed=SEED))



=== AFTER LoRA FINE-TUNE ===
------------------------------------------------------------
ROMEO:

I will, O my lord, be your master.

PAUL:
Then, you're my lord, I beg thee, to join the army!
O my lord, to you, I call
this town's king: here I beg thee, to
become my new master, if I should not
become yours, by some secret deed,
by
------------------------------------------------------------
To be, or not to be, and to be.

LORDY:
To be!

Gentlemen, you see there is not so much a difference between those things as to be.

LORDY:
Yes, I am very much pleased at the fact:
I know there is.

Gentlemen, you see I am no longer,
like all my peers.

------------------------------------------------------------
Once upon a time in fair Verona, and she shall bring forth a bride.

ROME:
With what honour I have found thee, O Romeo?

ROMEO:
And the more good I have done to thee, the more I may
be done with thee.

ROME:

O Romeo; I will go in and offer the roses of heaven,
Which are dear to me.



In [17]:
ppl_shakes_after = compute_ppl(model, val_ds)
ppl_control_after = compute_ppl(model, control_ds)
print(f"{'metric':<30}{'before':>10}{'after':>10}{'delta':>10}")
print(f"{'Shakespeare-val PPL':<30}{ppl_shakes_before:>10.2f}{ppl_shakes_after:>10.2f}{ppl_shakes_after - ppl_shakes_before:>+10.2f}")
print(f"{'Control (P&P) PPL':<30}{ppl_control_before:>10.2f}{ppl_control_after:>10.2f}{ppl_control_after - ppl_control_before:>+10.2f}")


metric                            before     after     delta
Shakespeare-val PPL                93.53     41.58    -51.94
Control (P&P) PPL                  16.92     18.62     +1.71


**Q2. (5 points)** Qualitatively, how did the outputs change? Mention at least two specific shifts you can point to (vocabulary, punctuation/structure, character names, line breaks, etc.).

**MY ANSWER**

## Stylistic shifts after fine-tuning
- Shift 1 (vocabulary / diction): The baseline outputs were generic modern prose about coaches, moral philosophy, and villages. After LoRA fine-tuning, the same prompts shifted toward theatrical diction: `O my lord`, `I beg thee`, `honour`, `Romeo`, `bride`, `roses of heaven`, and `king` all appear in the saved samples.
- Shift 2 (formatting / structure): The post-LoRA samples became much more play-like. They use speaker labels and line breaks such as `PAUL:`, `LORDY:`, `ROME:`, and `ROMEO:` instead of the baseline's paragraph prose.
- Shift 3 (named entities / setting): The `Once upon a time in fair Verona,` prompt now continues with Romeo/Verona-style material (`ROME`, `ROMEO`, `O Romeo`) rather than drifting into an unrelated village narrative.

**Q3. (5 points)** Report the four PPL numbers above. Did the control PPL move? In which direction, and by how much relative to the Shakespeare drop? What does this tell you about the trade-off you're making with LoRA fine-tuning?

**MY ANSWER**

## PPL numbers
| metric | before | after | delta |
|---|---:|---:|---:|
| Shakespeare-val PPL | 93.53 | 41.58 | -51.94 |
| Control (P&P) PPL | 16.92 | 18.62 | +1.71 |

## Direction and magnitude
- Control PPL moved up by `+1.71`, from `16.92` to `18.62`.
- The control degradation is much smaller than the Shakespeare-val improvement: `+1.71` control PPL versus `-51.94` Shakespeare PPL, about `3.3%` of the Shakespeare drop in absolute PPL points.

## Trade-off interpretation
LoRA bought a strong Shakespeare specialization with a small general-English cost. The adapter made held-out Shakespeare much easier for the model while slightly worsening the Pride & Prejudice control set, which is the expected domain-specialization trade-off.


## 9 · TODO: Save and load just the adapter (5 points)

A LoRA adapter is tiny (megabytes). Below, save **only** the LoRA parameters to disk, then verify you can load them back into a fresh GPT-2 and reproduce the trained model's outputs.


In [18]:
ADAPTER_PATH = "lora_adapter.pt"

def save_adapter(model: nn.Module, path: str) -> None:
    adapter_state = {
        name: param.detach().cpu().clone()
        for name, param in model.named_parameters()
        if "lora_" in name
    }
    assert adapter_state, "No LoRA parameters found to save."
    torch.save(adapter_state, path)


def load_adapter(model: nn.Module, path: str) -> None:
    adapter_state = torch.load(path, map_location="cpu")
    model_params = dict(model.named_parameters())

    for name, tensor in adapter_state.items():
        assert name in model_params, f"Saved adapter key {name!r} does not exist in this model."
        param = model_params[name]
        assert param.shape == tensor.shape, f"Shape mismatch for {name}: saved {tuple(tensor.shape)}, model {tuple(param.shape)}"
        with torch.no_grad():
            param.copy_(tensor.to(device=param.device, dtype=param.dtype))


save_adapter(model, ADAPTER_PATH)
print(f"Adapter file size: {os.path.getsize(ADAPTER_PATH) / 1e6:.2f} MB")



Adapter file size: 3.27 MB


In [19]:
# Round-trip check: build a fresh GPT-2 + LoRA, load the adapter, confirm outputs match.
fresh = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)
replace_conv1d_with_linear(fresh)
inject_lora(fresh, target_names=("c_attn", "c_proj"), r=8, alpha=16, dropout=0.05)
freeze_non_lora(fresh)
load_adapter(fresh, ADAPTER_PATH)
assert_same_device(fresh)

# IMPORTANT: a freshly-constructed nn.Module defaults to train mode, which leaves
# both GPT-2's own dropouts (attn_pdrop, resid_pdrop) and LoRALinear's dropout
# active. torch.no_grad() turns off autograd but NOT dropout — that's controlled
# by module.training. So set both models to eval before comparing logits.
model.eval()
fresh.eval()

with torch.no_grad():
    ids = tokenizer("ROMEO:", return_tensors="pt").input_ids.to(device)
    a = model(ids).logits
    b = fresh(ids).logits

print("max abs diff:", (a - b).abs().max().item())
assert torch.allclose(a, b, atol=1e-5), "Adapter round-trip failed."
print("OK: adapter round-trips.")


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 7805.51it/s]


max abs diff: 0.0
OK: adapter round-trips.


## 10 · TODO: Compare with the `peft` library (5 points)

Re-run the same fine-tune using `peft.LoraConfig` + `get_peft_model` on a fresh GPT-2 (with the original `Conv1D` layers — `peft` knows how to handle them). Match the hyperparameters: `r=8`, `alpha=16`, `dropout=0.05`, target modules `c_attn` and `c_proj`.


In [20]:
from peft import LoraConfig, get_peft_model, TaskType

peft_base = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["c_attn", "c_proj"],
    bias="none",
)
peft_model = get_peft_model(peft_base, peft_config).to(device)  # belt-and-suspenders: ensure adapter weights land on GPU
peft_model.print_trainable_parameters()
assert_same_device(peft_model)



Loading weights: 100%|██████████| 148/148 [00:00<00:00, 13599.37it/s]


trainable params: 811,008 || all params: 125,250,816 || trainable%: 0.6475


/Users/akhabalov-da_1/Documents/STUDY/nebius-ai-performance-engineering/code/lm-arch-ht4/.venv/lib/python3.14/site-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [21]:
peft_history = train_lora(peft_model, train_ds, val_ds, cfg)



epoch 0 step    0 | train loss 4.0903 | lr 2.00e-05
epoch 0 step   50 | train loss 4.0588 | lr 2.89e-04
epoch 0 step  100 | train loss 4.0121 | lr 2.43e-04
epoch 0 step  150 | train loss 3.8888 | lr 1.71e-04
== epoch 0 val loss 3.7219 (ppl 41.34) ==
epoch 1 step  200 | train loss 3.8431 | lr 9.39e-05
epoch 1 step  250 | train loss 3.8569 | lr 3.17e-05
epoch 1 step  300 | train loss 4.0146 | lr 1.40e-06
== epoch 1 val loss 3.6991 (ppl 40.41) ==


In [22]:
# Generate from the PEFT-trained model with the same prompts/seed.
print("=== PEFT-trained ===")
backup = model
model = peft_model  # so generate() uses it
for p in PROMPTS:
    print("-" * 60)
    print(generate(p, seed=SEED))
model = backup


=== PEFT-trained ===
------------------------------------------------------------
ROMEO:

I will, O my lord, be your master, and that your kingdom shall be secure; and I shall not lose your crown
If I should have been your king.

LUCIA:
But now, my lord, if he had been your king;
If he had not been your king, he would not have been,
Had he not been your king,
------------------------------------------------------------
To be, or not to be, and not to be,
And what kind of me, who was that, I will have
With you, and what kind of me
Who is that, to whom I have been pleased, to you,
For I am not a man of many, nor a woman;
And what kind of me, if I should have
Be a man, to you, as I have been
------------------------------------------------------------
Once upon a time in fair Verona, and she shall bring forth a bride.

ROME:
With what honour I have found thee, O Romeo?

ROMEO:
And the more good I have done to thee, the more I may
be done with thee.

ROME:

When she shall bring forth the b

In [23]:
# Quantitative comparison: PEFT vs hand-rolled, on the same val/control sets.
ppl_shakes_peft = compute_ppl(peft_model, val_ds)
ppl_control_peft = compute_ppl(peft_model, control_ds)
print(f"{'metric':<30}{'hand-rolled':>14}{'PEFT':>10}")
print(f"{'Shakespeare-val PPL':<30}{ppl_shakes_after:>14.2f}{ppl_shakes_peft:>10.2f}")
print(f"{'Control (P&P) PPL':<30}{ppl_control_after:>14.2f}{ppl_control_peft:>10.2f}")


metric                           hand-rolled      PEFT
Shakespeare-val PPL                    41.58     41.45
Control (P&P) PPL                      18.62     18.85


**Q4.(5 points)** Compare your hand-rolled trainable parameter count vs `peft_model.print_trainable_parameters()`. Are they identical? If not, where does the difference come from? (Hint: think about which projections `peft` decides to wrap given `target_modules=["c_attn", "c_proj"]`.)

**MY ANSWER**

## Param counts
| implementation | wrapped modules | trainable params | total params | trainable % |
|---|---:|---:|---:|---:|
| Hand-rolled LoRA | 36 | 811,008 | about 125.3M | about 0.65% |
| PEFT LoRA | 36 | 811,008 | 125,250,816 | 0.6475% |

Yes, the trainable parameter counts are identical: both implementations train exactly `811,008` LoRA parameters.

## Why they match
Both implementations target the same GPT-2 module names: `c_attn` and `c_proj`. In each of the 12 transformer blocks, this wraps three projections:

- `attn.c_attn`, the fused QKV projection, shape `768 -> 2304`: `8 * (768 + 2304) = 24,576` LoRA params
- `attn.c_proj`, the attention output projection, shape `768 -> 768`: `8 * (768 + 768) = 12,288` LoRA params
- `mlp.c_proj`, the MLP output/down projection, shape `3072 -> 768`: `8 * (3072 + 768) = 30,720` LoRA params

Per block this is `24,576 + 12,288 + 30,720 = 67,584` trainable params. Across 12 blocks: `67,584 * 12 = 811,008`.

PEFT suffix-matches the same module names, so `target_modules=["c_attn", "c_proj"]` hits the same 36 projections. The only implementation difference is that PEFT wraps GPT-2's original `Conv1D` modules and sets `fan_in_fan_out=True`, while the hand-rolled version first converts those modules to `nn.Linear`. That changes the internal weight orientation handling, not the LoRA parameter count.

**Q5.(5 points)** After the same number of training steps, did your hand-rolled LoRA reach a similar Shakespeare-val PPL to the PEFT version? List one reason they could differ even with matched hyperparameters.

**MY ANSWER**

## Shakespeare-val PPL comparison
| metric | hand-rolled LoRA | PEFT LoRA | absolute difference |
|---|---:|---:|---:|
| Shakespeare-val PPL | 41.58 | 41.45 | 0.13 |
| Control (P&P) PPL | 18.62 | 18.85 | 0.23 |

Yes, the Shakespeare validation PPL is effectively the same. The PEFT run is slightly lower (`41.45` vs `41.58`), a difference of only about `0.31%`, which is well inside a 10% similarity band.

## Why they can still differ
Even with matched `r`, `alpha`, dropout, target modules, optimizer, and training schedule, the two runs are not guaranteed to be bit-identical. The adapters are initialized separately, the dataloader shuffle order can differ, dropout samples can differ, and PEFT handles GPT-2 `Conv1D` layers directly with `fan_in_fan_out=True` while the hand-rolled path trains equivalent LoRA matrices after converting those projections to `nn.Linear`. Those small implementation and randomness differences are enough to move PPL by a fraction of a point.



## 11 · Bonus: pick a different style

Pick one (or all three!) and watch the model become a different character. Each option below builds a `raw_text` string; everything downstream of `LMChunks(raw_text, …)` is identical.

> **Tip:** for any of these, **reset the model** before the new fine-tune (re-run cell 5 first), or you'll be training a Shakespeare-flavoured Rick. Or do that on purpose. It's funny.

**How this notebook implements the bonus:**

Each option builds a character/style corpus as a `raw_text_*` string. The clean experiments inside 11A, 11B, and 11C each start from fresh GPT-2 + fresh LoRA and train only on that corpus, which follows the assignment's intended path:

```text
raw_text_* -> LMChunks(raw_text_*, tokenizer, BLOCK_SIZE) -> fresh LoRA -> style fine-tune -> first-400-char samples
```

| option | clean experiment | start adapter | corpus |
|---|---|---|---|
| 11A | Rick with Rick | fresh LoRA | `raw_text_rick` |
| 11B | Hermione with Hermione | fresh LoRA | `raw_text_hermione` |
| 11C | Yoda with Yoda | fresh LoRA | `raw_text_yoda` |
| 11D | parameterized hybrid | fresh or Shakespeare adapter | style corpus + content prompt |

11D is an extra controlled variant of the same idea: it keeps style text and content prompt separate. For example, Yoda-style generated text can speak from a Shakespeare prompt, or Shakespeare-style generated text can speak from a Yoda prompt. To avoid another long accidental run, 11D is capped as a micro fine-tune and is disabled by default.



### Shared bonus helpers

These helpers keep the bonus experiments consistent. They build a fresh GPT-2 + LoRA model, build `LMChunks` from the selected `raw_text_*`, fine-tune only LoRA parameters, and print first-400-character samples.


In [24]:
# Shared helpers for clean and hybrid bonus experiments.
def make_lora_model(start_from: str = "fresh") -> nn.Module:
    """Build GPT-2 + LoRA. Optionally load the saved Shakespeare adapter."""
    bonus_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)
    replace_conv1d_with_linear(bonus_model)
    inject_lora(bonus_model, target_names=("c_attn", "c_proj"), r=8, alpha=16, dropout=0.05)
    freeze_non_lora(bonus_model)

    if start_from == "shakespeare":
        assert os.path.exists(ADAPTER_PATH), f"{ADAPTER_PATH} not found. Run the adapter save cell first."
        load_adapter(bonus_model, ADAPTER_PATH)
    elif start_from != "fresh":
        raise ValueError("start_from must be either 'fresh' or 'shakespeare'")

    assert_same_device(bonus_model)
    return bonus_model


def make_style_datasets(
    style_text: str,
    max_style_chars: Optional[int] = None,
    max_train_chunks: Optional[int] = None,
    max_val_chunks: Optional[int] = None,
):
    """Use the same raw_text -> LMChunks path, optionally capped for fast bonus demos."""
    assert style_text.strip(), "Style corpus is empty. Run/fix the corpus loading cell first."
    if max_style_chars is not None:
        style_text = style_text[:max_style_chars]

    full_ds = LMChunks(style_text, tokenizer, BLOCK_SIZE)
    n_val = max(1, len(full_ds) // 20)
    train_ds, val_ds = torch.utils.data.random_split(
        full_ds,
        [len(full_ds) - n_val, n_val],
        generator=torch.Generator().manual_seed(SEED),
    )

    if max_train_chunks is not None:
        train_ds = torch.utils.data.Subset(train_ds, range(min(max_train_chunks, len(train_ds))))
    if max_val_chunks is not None:
        val_ds = torch.utils.data.Subset(val_ds, range(min(max_val_chunks, len(val_ds))))
    return train_ds, val_ds


@torch.no_grad()
def generate_with_model(active_model: nn.Module, prompt: str, max_new_tokens: int = 120, seed: int = 0) -> str:
    torch.manual_seed(seed)
    active_model.eval()
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    out = active_model.generate(
        ids,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.9,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)


def run_style_experiment(
    name: str,
    style_text: str,
    prompts: list[str],
    start_from: str = "fresh",
    epochs: int = 1,
    train_cfg: Optional[TrainCfg] = None,
    max_style_chars: Optional[int] = None,
    max_train_chunks: Optional[int] = None,
    max_val_chunks: Optional[int] = None,
    max_new_tokens: int = 120,
) -> dict:
    """Fine-tune a LoRA adapter on one style corpus and print first-400-char samples."""
    preview_text = style_text[: max_style_chars or len(style_text)]
    print(f"=== {name}: corpus preview, first 400 chars ===")
    print(preview_text[:400])

    style_train_ds, style_val_ds = make_style_datasets(
        style_text,
        max_style_chars=max_style_chars,
        max_train_chunks=max_train_chunks,
        max_val_chunks=max_val_chunks,
    )
    print(f"{name} train chunks: {len(style_train_ds)} | val chunks: {len(style_val_ds)} | block_size={BLOCK_SIZE}")

    style_model = make_lora_model(start_from=start_from)
    print(f"=== {name}: before style fine-tune ({start_from}), first 400 chars ===")
    for prompt in prompts:
        print("-" * 60)
        print(generate_with_model(style_model, prompt, max_new_tokens=max_new_tokens, seed=SEED)[:400])

    style_cfg = train_cfg or TrainCfg(epochs=epochs, batch_size=8, lr=3e-4, log_every=25)
    style_history = train_lora(style_model, style_train_ds, style_val_ds, style_cfg)

    print(f"=== {name}: after style fine-tune, first 400 chars ===")
    samples = {}
    for offset, prompt in enumerate(prompts):
        print("-" * 60)
        sample = generate_with_model(style_model, prompt, max_new_tokens=max_new_tokens, seed=SEED + offset)[:400]
        samples[prompt] = sample
        print(sample)
    return {"model": style_model, "history": style_history, "train_ds": style_train_ds, "val_ds": style_val_ds, "samples": samples}




### 11A · Rick & Morty

Builds `raw_text_rick` from Rick dialogue in a community Hugging Face transcript. The clean experiment below trains Rick with Rick: fresh GPT-2 + fresh LoRA, using only `raw_text_rick`.


In [25]:
# Bonus A: Rick & Morty dialogue.
# This uses the `Prarabdha/Rick_and_Morty_Transcript` dataset on HF Hub.
# If that dataset is unavailable, swap in any other dialogue source — what matters is the format.
from datasets import load_dataset

try:
    ds = load_dataset("Prarabdha/Rick_and_Morty_Transcript", split="train")
    # Inspect if the dataset schema changes:
    # print(ds.column_names, ds[0])

    # Some community rows contain None values, so normalize before calling string methods.
    rick_lines = []
    for row in ds:
        speaker = str(row.get("speaker") or "").strip().lower()
        line = row.get("dialouge") or row.get("dialogue") or row.get("line") or ""
        line = str(line).strip()
        if speaker == "rick" and line:
            rick_lines.append(line)

    raw_text_rick = "\n".join(f"Rick: {line}" for line in rick_lines)
    print(f"Rick corpus: {len(raw_text_rick):,} chars over {len(rick_lines)} lines")
    print(raw_text_rick[:400])
except Exception as e:
    print("Dataset load failed:", e)
    print("Fallback: paste your own transcript text into raw_text_rick.")
    raw_text_rick = ""

# To use this corpus, set `raw_text = raw_text_rick` and re-run the dataset / training cells.



Rick corpus: 170,356 chars over 1647 lines
Rick: stumbles in drunkenly, and turns on the lights. Morty! You gotta
                    come on. Jus'... you gotta come with me.
Rick: I got a surprise for you, Morty.
Rick: spills alcohol on Morty's bed Come on, I got a surprise for you. drags
                        Morty by the ankle Come on, hurry up. pulls Morty out of his bed and into
                        the hall.
Rick: We gotta go, g


#### Clean 11A experiment: Rick with Rick

Set `RUN_CLEAN_RICK = True` to train a fresh LoRA adapter only on `raw_text_rick` and print first-400-character samples.


In [32]:
# Clean 11A: Rick with Rick.
RUN_CLEAN_RICK = True

if RUN_CLEAN_RICK:
    rick_result = run_style_experiment(
        "11A / Rick with Rick",
        raw_text_rick,
        prompts=["Rick:", "Morty:", "Listen, Morty,"],
        start_from="fresh",
        epochs=1,
    )



=== 11A / Rick with Rick: corpus preview, first 400 chars ===
Rick: stumbles in drunkenly, and turns on the lights. Morty! You gotta
                    come on. Jus'... you gotta come with me.
Rick: I got a surprise for you, Morty.
Rick: spills alcohol on Morty's bed Come on, I got a surprise for you. drags
                        Morty by the ankle Come on, hurry up. pulls Morty out of his bed and into
                        the hall.
Rick: We gotta go, g
11A / Rick with Rick train chunks: 195 | val chunks: 10 | block_size=256


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 8122.65it/s]


=== 11A / Rick with Rick: before style fine-tune (fresh), first 400 chars ===
------------------------------------------------------------
Rick: I did hear about the story with Jim and I was like, "What about that?"


Diana: He's such a gentleman. He's like the president of the United States. He can do whatever he wants with the military.


Gary: I would say it's true. I think that with all the money that's gone into it, he actually got one job. He got a position at an Air Force base in Ohio where he is in charge of making sure that 
------------------------------------------------------------
Morty: I did something I thought was stupid. It was an action film!

Gavin: Yeah, like there was some way I was gonna do that.

Morris: You were saying to yourself, like, 'I don't like these guys. I would rather die in the way I died in the film.'

Gavin: That's a real joke.

Morris: Like if I say it right, then I'll do it.

Gavin: It's like I'm just thinking, you know what, I don't like these gu

### 11B · Hermione Granger

Builds `raw_text_hermione` from `yuyouyu/BeyondDialogue`. The clean experiment below trains Hermione with Hermione: fresh GPT-2 + fresh LoRA, using only `raw_text_hermione`.


In [33]:
# Bonus B: Hermione Granger dialogue.
# This uses the `yuyouyu/BeyondDialogue` dataset on HF Hub.
# The file is character-specific: ChunkDialogues_EN/ Harry Potter/Hermione.json
from huggingface_hub import hf_hub_download
import json


def collect_text(obj):
    """Recursively collect string leaves from a nested JSON object."""
    texts = []
    if isinstance(obj, str):
        texts.append(obj)
    elif isinstance(obj, dict):
        for value in obj.values():
            texts.extend(collect_text(value))
    elif isinstance(obj, list):
        for item in obj:
            texts.extend(collect_text(item))
    return texts


try:
    hermione_path = hf_hub_download(
        repo_id="yuyouyu/BeyondDialogue",
        repo_type="dataset",
        filename="ChunkDialogues_EN/ Harry Potter/Hermione.json",
    )

    with open(hermione_path, "r", encoding="utf-8") as f:
        hermione_data = json.load(f)

    hermione_lines = []
    seen = set()
    for text_item in collect_text(hermione_data):
        line = " ".join(str(text_item).split())
        # Keep dialogue-like snippets and avoid tiny metadata fragments.
        if len(line) >= 20 and line not in seen:
            seen.add(line)
            hermione_lines.append(line)

    raw_text_hermione = "\n".join(f"Hermione: {line}" for line in hermione_lines)
    print(f"Hermione corpus: {len(raw_text_hermione):,} chars over {len(hermione_lines)} lines")
    print(raw_text_hermione[:400])
except Exception as e:
    print("Hermione dataset load failed:", e)
    print("Fallback: paste your own Hermione dialogue text into raw_text_hermione.")
    raw_text_hermione = ""

# To use this corpus, set `raw_text = raw_text_hermione` and re-run the dataset / training cells.



Hermione corpus: 1,983,299 chars over 2021 lines
Hermione: Hermione Jean Granger
Hermione: 问题多小姐,十全十美小姐,赫—米—恩,万事通,赫米,大板牙,赫—米—翁
Hermione: Helpful,Rational,Clever,Just
Hermione: 20th Century Magical World
Hermione: “He might have died and you wouldn't know the difference,” “I tried to turn him yellow yesterday to make him more interesting, but the spell didn't work. I'll show you, look . . .” “Unicorn hair's nearly poking out. Anyway —”
Hermione: 


#### Clean 11B experiment: Hermione with Hermione

Set `RUN_CLEAN_HERMIONE = True` to train a fresh LoRA adapter only on `raw_text_hermione` and print first-400-character samples.


In [34]:
# Clean 11B: Hermione with Hermione.
RUN_CLEAN_HERMIONE = True

if RUN_CLEAN_HERMIONE:
    hermione_result = run_style_experiment(
        "11B / Hermione with Hermione",
        raw_text_hermione,
        prompts=["Hermione:", "Harry,", "Ron,"],
        start_from="fresh",
        epochs=1,
    )



=== 11B / Hermione with Hermione: corpus preview, first 400 chars ===
Hermione: Hermione Jean Granger
Hermione: 问题多小姐,十全十美小姐,赫—米—恩,万事通,赫米,大板牙,赫—米—翁
Hermione: Helpful,Rational,Clever,Just
Hermione: 20th Century Magical World
Hermione: “He might have died and you wouldn't know the difference,” “I tried to turn him yellow yesterday to make him more interesting, but the spell didn't work. I'll show you, look . . .” “Unicorn hair's nearly poking out. Anyway —”
Hermione: 
11B / Hermione with Hermione train chunks: 1721 | val chunks: 90 | block_size=256


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 6782.45it/s]


=== 11B / Hermione with Hermione: before style fine-tune (fresh), first 400 chars ===
------------------------------------------------------------
Hermione: I did hear something, Hermione. I didn't know what happened. I didn't know if she'd like it or not. I was worried we wouldn't have to wait too long. I thought to myself, well, I just don't think we should have waited too long.

"Hermione, you really should think about what you've just said to him. We should just let him know you did something wrong, right?"

Harry nodded, nodding again.

------------------------------------------------------------
Harry, and she wanted to get better with her voice. That's when her voice started to grow.

"And you know what I mean."

"I'm not doing anything. I'm talking to you like a kid."

"I understand. I know that's a bad way to put it, but I think the one you're talking about is actually a good way to help me cope with my depression, it doesn't hurt to take care of yourself. It's a relief to lea

### 11C · Yoda

Builds `raw_text_yoda` from an inline quote list. The clean experiment below trains Yoda with Yoda: fresh GPT-2 + fresh LoRA, using only `raw_text_yoda`.


In [35]:
# Bonus C: Yoda. Inline starter corpus — feel free to extend.
yoda_quotes = [
    "Do or do not. There is no try.",
    "Fear is the path to the dark side. Fear leads to anger. Anger leads to hate. Hate leads to suffering.",
    "Size matters not. Look at me. Judge me by my size, do you?",
    "When nine hundred years old you reach, look as good you will not.",
    "Wars not make one great.",
    "Adventure. Heh. Excitement. Heh. A Jedi craves not these things.",
    "Patience you must have, my young padawan.",
    "Truly wonderful, the mind of a child is.",
    "Always pass on what you have learned.",
    "Train yourself to let go of everything you fear to lose.",
    "Difficult to see. Always in motion is the future.",
    "The greatest teacher, failure is.",
    "Pass on what you have learned. Strength. Mastery.",
    "Named must your fear be before banish it you can.",
    "Powerful you have become, the dark side I sense in you.",
    "You must unlearn what you have learned.",
    "That is why you fail.",
    "Once you start down the dark path, forever will it dominate your destiny.",
    "Mind what you have learned. Save you it can.",
    "Luminous beings are we, not this crude matter.",
    "Through the Force, things you will see. Other places. The future, the past. Old friends long gone.",
    "Decide you must, how to serve them best.",
    "Hard to see, the dark side is.",
    "Strong with the Force, young Skywalker is.",
    "Many of the truths that we cling to depend on our point of view.",
    "Smaller in number are we, but larger in mind.",
    "Around the survivors a perimeter create.",
    "Rejoice for those around you who transform into the Force.",
    "Mourn them, do not. Miss them, do not.",
    "When you look at the dark side, careful you must be.",
    "Help you I can. Yes. Mmm.",
    "Already know you that which you need.",
    "A Jedi's strength flows from the Force.",
    "Anger, fear, aggression. The dark side are they.",
    "If once you start down the dark path, forever will it dominate your destiny.",
    "Consume you it will, as it did Obi-Wan's apprentice.",
    "Do not underestimate the powers of the Emperor or suffer your father's fate, you will.",
    "Reckless he is. Matters are worse.",
    "Control, control. You must learn control.",
    "Yes, a Jedi's strength flows from the Force. But beware. Anger, fear, aggression.",
    "Stopped they must be. On this all depends.",
    "Faith in your friends, yours is.",
    "Soon will I rest. Yes, forever sleep. Earned it I have.",
    "Twilight is upon me, and soon night must fall.",
    "When gone am I, the last of the Jedi will you be.",
    "The Force runs strong in your family. Pass on what you have learned.",
    "There is another. Sky-walker.",
    "No. Try not. Do, or do not. There is no try.",
    "Judge me by my size, do you?",
    "Concentrate. Feel the Force flow.",
]

# Repeat to give the model enough tokens to chunk into BLOCK_SIZE windows.
raw_text_yoda = ("\n".join(yoda_quotes) + "\n") * 200
print(f"Yoda corpus: {len(raw_text_yoda):,} chars (after replication)")
print(raw_text_yoda[:300])



Yoda corpus: 495,200 chars (after replication)
Do or do not. There is no try.
Fear is the path to the dark side. Fear leads to anger. Anger leads to hate. Hate leads to suffering.
Size matters not. Look at me. Judge me by my size, do you?
When nine hundred years old you reach, look as good you will not.
Wars not make one great.
Adventure. Heh. E


#### Clean 11C experiment: Yoda with Yoda

Set `RUN_CLEAN_YODA = True` to train a fresh LoRA adapter only on `raw_text_yoda` and print first-400-character samples.


In [36]:
# Clean 11C: Yoda with Yoda.
RUN_CLEAN_YODA = True

if RUN_CLEAN_YODA:
    yoda_result = run_style_experiment(
        "11C / Yoda with Yoda",
        raw_text_yoda,
        prompts=["Yoda:", "Train yourself", "The Force"],
        start_from="fresh",
        epochs=1,
    )



=== 11C / Yoda with Yoda: corpus preview, first 400 chars ===
Do or do not. There is no try.
Fear is the path to the dark side. Fear leads to anger. Anger leads to hate. Hate leads to suffering.
Size matters not. Look at me. Judge me by my size, do you?
When nine hundred years old you reach, look as good you will not.
Wars not make one great.
Adventure. Heh. Excitement. Heh. A Jedi craves not these things.
Patience you must have, my young padawan.
Truly wond
11C / Yoda with Yoda train chunks: 492 | val chunks: 25 | block_size=256


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 6855.63it/s]


=== 11C / Yoda with Yoda: before style fine-tune (fresh), first 400 chars ===
------------------------------------------------------------
Yoda: I did something I thought was stupid. It was an action film!

Gotham City: That's how you learn to love yourself. You don't know what you're doing until you can't help it, like watching another family in a room. I used to feel like if you were able to go in and out of the room, you'd get some of the things you want to happen. You're always going to find something that works for you.

Jyn: It
------------------------------------------------------------
Train yourself and your team will get better with time and practice.

You should not worry about getting injured or being injured in practice

When you're out of the game, you have a chance to show your teammates what you can do on the road. If you are practicing and feel confident, you can impress your team in the long run. If you play, get ready to win, and be proud of your work ethic.

Be awar

In [37]:
# Q6 exact forgetting metrics for the clean bonus adapters.
# These reuse the main Shakespeare validation set and Pride & Prejudice control set.
bonus_results = {
    "Rick": rick_result,
    "Hermione": hermione_result,
    "Yoda": yoda_result,
}

q6_bonus_ppl = {}
print(f"{'bonus run':<12}{'style-val':>12}{'shakes-val':>14}{'control':>12}{'shake delta':>14}{'control delta':>15}")
for name, result in bonus_results.items():
    bonus_model = result["model"]
    last_style_val = result["history"]["val_loss"][-1]
    style_loss = last_style_val[1] if isinstance(last_style_val, tuple) else last_style_val
    style_ppl = math.exp(style_loss)
    shakes_ppl = compute_ppl(bonus_model, val_ds)
    control_ppl = compute_ppl(bonus_model, control_ds)
    q6_bonus_ppl[name] = {
        "style_val_ppl": style_ppl,
        "shakespeare_val_ppl": shakes_ppl,
        "control_ppl": control_ppl,
        "shakespeare_delta_vs_baseline": shakes_ppl - ppl_shakes_before,
        "control_delta_vs_baseline": control_ppl - ppl_control_before,
    }
    print(
        f"{name:<12}{style_ppl:>12.2f}{shakes_ppl:>14.2f}{control_ppl:>12.2f}"
        f"{shakes_ppl - ppl_shakes_before:>+14.2f}{control_ppl - ppl_control_before:>+15.2f}"
    )

print("\nBaseline for deltas:")
print(f"Shakespeare-val PPL: {ppl_shakes_before:.2f}")
print(f"Control PPL: {ppl_control_before:.2f}")


bonus run      style-val    shakes-val     control   shake delta  control delta
Rick               26.08         77.26       17.01        -16.26          +0.09
Hermione           20.57         90.89       18.53         -2.64          +1.61
Yoda               11.61        123.61       26.70        +30.08          +9.79

Baseline for deltas:
Shakespeare-val PPL: 93.53
Control PPL: 16.92


### 11D · Parameterized hybrid style/content + Rime TTS

This block lets you combine a style corpus with a separate content prompt. Examples:

- Yoda style speaking Shakespeare text: `STYLE_NAME="Yoda"`, `STYLE_TEXT=raw_text_yoda`, `CONTENT_PROMPT="To be, or not to be,"`.
- Shakespeare style speaking Yoda text: `STYLE_NAME="Shakespeare"`, `STYLE_TEXT=raw_text`, `CONTENT_PROMPT="Do or do not. There is no try."`.

To keep 11D interesting but cheap, it runs as a **micro fine-tune** when enabled: it slices the style corpus, trains on a small number of chunks, and generates a short hybrid sample. This still shows the adapter bending the model toward a style, but avoids another long full GPT-2 training pass during `Run All`.

The optional Rime step sends the generated 11D sample to TTS. It reads `RIME_TOKEN` from `.env` or the environment and writes an audio file under `outputs/`.

Voice presets selected from Rime's current voice metadata:

- `yoda`: `orion` on `arcana` — English elder male voice, a better fit for wise Yoda-style output than the earlier generic `celeste` voice.
- `shakespeare`: `albion` on `arcana` — English male voice from England and marked as a flagship voice, a better fit for theatrical Shakespeare-style output.



In [37]:
# 11D: parameterized hybrid style/content micro-experiment with optional Rime TTS.
from pathlib import Path
import json as _json
import urllib.request

RIME_VOICE_PRESETS = {
    "yoda": {
        "speaker": "orion",
        "model_id": "arcana",
        "why": "English elder male Arcana voice; best available fit for wise Yoda-style delivery.",
    },
    "shakespeare": {
        "speaker": "albion",
        "model_id": "arcana",
        "why": "English male Arcana flagship voice from England; best fit for theatrical Shakespeare-style delivery.",
    },
}

# Fast 11D defaults: 48 chunks / batch 8 = about 6 optimizer steps, plus 8 validation chunks.
FAST_11D_TRAIN_CFG = TrainCfg(epochs=1, batch_size=8, lr=5e-4, log_every=1)
FAST_11D_MAX_STYLE_CHARS = 60_000
FAST_11D_MAX_TRAIN_CHUNKS = 48
FAST_11D_MAX_VAL_CHUNKS = 8
FAST_11D_MAX_NEW_TOKENS = 90


def load_dotenv_value(key: str, path: str = ".env") -> str:
    if key in os.environ and os.environ[key].strip():
        return os.environ[key].strip()
    env_path = Path(path)
    if not env_path.exists():
        return ""
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        if k.strip() == key:
            return v.strip().strip("\"'")
    return ""


def synthesize_rime_tts(text: str, output_path: str, speaker: str, model_id: str = "arcana", token_env: str = "RIME_TOKEN") -> str:
    token = load_dotenv_value(token_env) or load_dotenv_value("RIME_API_KEY")
    assert token, f"Set {token_env} in .env or the environment before running Rime TTS."
    headers = {"Accept": "audio/mp3", "Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    payload = {"text": text, "speaker": speaker, "modelId": model_id}
    request = urllib.request.Request("https://users.rime.ai/v1/rime-tts", data=_json.dumps(payload).encode("utf-8"), headers=headers, method="POST")
    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen(request) as response:
        with output.open("wb") as f:
            while chunk := response.read(4096):
                f.write(chunk)
    return str(output)


def run_hybrid_experiment_with_tts(
    style_name: str,
    style_text: str,
    content_prompt: str,
    start_from: str = "fresh",
    voice_preset: str = "yoda",
    run_tts: bool = False,
    train_cfg: TrainCfg = FAST_11D_TRAIN_CFG,
    max_style_chars: int = FAST_11D_MAX_STYLE_CHARS,
    max_train_chunks: int = FAST_11D_MAX_TRAIN_CHUNKS,
    max_val_chunks: int = FAST_11D_MAX_VAL_CHUNKS,
    max_new_tokens: int = FAST_11D_MAX_NEW_TOKENS,
) -> dict:
    result = run_style_experiment(
        f"11D micro / {style_name} style over content prompt",
        style_text,
        prompts=[content_prompt],
        start_from=start_from,
        train_cfg=train_cfg,
        max_style_chars=max_style_chars,
        max_train_chunks=max_train_chunks,
        max_val_chunks=max_val_chunks,
        max_new_tokens=max_new_tokens,
    )
    generated_text = result["samples"][content_prompt]
    result["generated_text"] = generated_text
    print("=== 11D generated text for TTS: first 400 chars ===")
    print(generated_text[:400])

    preset = RIME_VOICE_PRESETS[voice_preset]
    result["rime_voice"] = preset
    print(f"Rime voice preset: {voice_preset} -> speaker={preset['speaker']} model_id={preset['model_id']}")
    print(preset["why"])

    if run_tts:
        audio_path = synthesize_rime_tts(
            generated_text[:700],
            output_path=f"outputs/rime_11d_{style_name.lower().replace(' ', '_')}_{voice_preset}_{preset['speaker']}.mp3",
            speaker=preset["speaker"],
            model_id=preset["model_id"],
        )
        result["audio_path"] = audio_path
        print(f"Rime TTS audio saved to: {audio_path}")
    return result


# Keep these False by default so Run All does not start another GPT-2 fine-tune.
# Turn on exactly one micro demo when you want to regenerate 11D outputs.
RUN_11D_YODA_STYLE_SHAKESPEARE_TEXT = True
RUN_11D_SHAKESPEARE_STYLE_YODA_TEXT = True
RUN_RIME_TTS = True

if RUN_11D_YODA_STYLE_SHAKESPEARE_TEXT:
    yoda_speaks_shakespeare = run_hybrid_experiment_with_tts(
        style_name="Yoda",
        style_text=raw_text_yoda,
        content_prompt="To be, or not to be, that is the question:",
        start_from="fresh",
        voice_preset="yoda",
        run_tts=RUN_RIME_TTS,
    )

if RUN_11D_SHAKESPEARE_STYLE_YODA_TEXT:
    shakespeare_speaks_yoda = run_hybrid_experiment_with_tts(
        style_name="Shakespeare",
        style_text=raw_text,  # TinyShakespeare corpus from section 6
        content_prompt="Do or do not. There is no try.",
        start_from="fresh",
        voice_preset="shakespeare",
        run_tts=RUN_RIME_TTS,
    )

# Pre-generated local demo files from the selected presets:
# - outputs/rime_11d_yoda_voice_orion_yoda_style_shakespeare.mp3
# - outputs/rime_11d_shakespeare_voice_albion_yoda_text.mp3




=== 11D micro / Yoda style over content prompt: corpus preview, first 400 chars ===
Do or do not. There is no try.
Fear is the path to the dark side. Fear leads to anger. Anger leads to hate. Hate leads to suffering.
Size matters not. Look at me. Judge me by my size, do you?
When nine hundred years old you reach, look as good you will not.
Wars not make one great.
Adventure. Heh. Excitement. Heh. A Jedi craves not these things.
Patience you must have, my young padawan.
Truly wond
11D micro / Yoda style over content prompt train chunks: 48 | val chunks: 3 | block_size=256


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 5771.96it/s]


=== 11D micro / Yoda style over content prompt: before style fine-tune (fresh), first 400 chars ===
------------------------------------------------------------
To be, or not to be, that is the question: is she going to get better with age? The answer is no!

Her performance is remarkable, and there is no way I can argue with that. Her talent has been remarkable. Her intelligence has been extraordinary. Her talent has always been outstanding. Her character is astounding and impressive in every way.

But if she had not been this hard, she would not have be
epoch 0 step    0 | train loss 3.8286 | lr 4.67e-04
epoch 0 step    1 | train loss 3.7517 | lr 3.75e-04
epoch 0 step    2 | train loss 3.7852 | lr 2.50e-04
epoch 0 step    3 | train loss 3.6659 | lr 1.25e-04
epoch 0 step    4 | train loss 3.7085 | lr 3.35e-05
epoch 0 step    5 | train loss 3.7460 | lr 0.00e+00
== epoch 0 val loss 3.5383 (ppl 34.41) ==
=== 11D micro / Yoda style over content prompt: after style fine-tune, first 400 cha

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 5620.25it/s]


=== 11D micro / Shakespeare style over content prompt: before style fine-tune (fresh), first 400 chars ===
------------------------------------------------------------
Do or do not. There is no try. The world of science, of science, of science... What you are asking about... is not possible. There's no 'try'. And, we know that it's impossible for people to find that to be true. It's just another form of madness. It is a form of madness. It is the disease of the human mind. It's the disease of the human soul. It is a sickness of the human psyche. It has no cure
epoch 0 step    0 | train loss 4.5892 | lr 4.67e-04
epoch 0 step    1 | train loss 4.6806 | lr 3.75e-04
epoch 0 step    2 | train loss 4.3919 | lr 2.50e-04
epoch 0 step    3 | train loss 4.5095 | lr 1.25e-04
epoch 0 step    4 | train loss 4.5023 | lr 3.35e-05
epoch 0 step    5 | train loss 4.5127 | lr 0.00e+00
== epoch 0 val loss 4.2427 (ppl 69.60) ==
=== 11D micro / Shakespeare style over content prompt: after style fine-tune, f

**Q6 (bonus).** Pick at least one of A/B/C/D, fine-tune, and paste 3 generated samples that show the style. Also report Shakespeare-val PPL and control PPL after this run — does Shakespeare PPL get worse? One sentence on what surprised you most.

**MY ANSWER**

## Bonus run setup
- Datasets chosen in the last full run: A / Rick, B / Hermione, and C / Yoda.
- Run shape: each clean bonus experiment used fresh GPT-2 + hand-rolled LoRA, `r=8`, `alpha=16`, and one focused epoch on its own `raw_text_*` corpus.
- Clean run metrics printed by the notebook: Rick `195` train chunks / `10` val chunks / style-val PPL `26.08`; Hermione `1721` train chunks / `90` val chunks / style-val PPL `20.57`; Yoda `492` train chunks / `25` val chunks / style-val PPL `11.61`.
- The Q6 metrics cell evaluates each clean bonus adapter on the main Shakespeare validation set and the Pride & Prejudice control set, using baseline PPLs `93.53` and `16.92` for deltas.
- Rime TTS demos also ran for the optional 11D micro experiments with Yoda voice `orion` and Shakespeare voice `albion`.

## Three generated samples that show the style
1. Rick, prompt `Listen, Morty,`  
   `Listen, Morty, I love you!" Morty said, "My God, your face is so beautiful! It is so beautiful when you get to sing to it!" Rick and Morty both looked at each other for a moment. "Rick, Morty, I love you!" Morty said, "My God, your face is so beautiful! It is so beautiful when you get to sing to it!" Rick sighed and said, "You're a good guy, Morty. Don't let this be the first time. Come back t`

2. Hermione, prompt `Hermione:`  
   `Hermione: ... Hermione: ...Hermione, did you think you knew that your name was Ron Weasley? Ron's father? Hermione: Did you know that he was now dead? Hermione, can you help me, Harry? Hermione: Harry, Ron, and Hermione are all in the same category, Hermione, Ron, Hermione, Hermione, Ron, Hermione, Ron, Hermione, Hermione, Ron, Hermione, Hermione, Ron, Hermione, Ron, Hermione, Ron, Hermione, Ron,`

3. Yoda, prompt `The Force`  
   `The Force is too great at work. When the Jedi are victorious the dark side must rise. The only good Jedi are those who follow their mentors. They must not. It is time to begin to guide the Force. The Force is the most powerful weapon of all. Learn to learn. It is. Learn to master the Force. Go on you go. I ask no more. To grow you will. I cannot. To do good you must. The Jedi must be. They must. T`

## PPL after the bonus fine-tune
| clean bonus run | style-val PPL | Shakespeare-val PPL | Control P&P PPL | Shakespeare delta vs baseline | Control delta vs baseline |
|---|---:|---:|---:|---:|---:|
| Rick with Rick | 26.08 | 77.26 | 17.01 | -16.26 | +0.09 |
| Hermione with Hermione | 20.57 | 90.89 | 18.53 | -2.64 | +1.61 |
| Yoda with Yoda | 11.61 | 123.61 | 26.70 | +30.08 | +9.79 |

Rick and Hermione did not make Shakespeare-val PPL worse than the untuned baseline, but both remained far worse than the Shakespeare LoRA adapter's `41.58` PPL. Yoda did make Shakespeare-val PPL worse, increasing it from `93.53` to `123.61`, and it also damaged the control set the most (`16.92` to `26.70`). The exact numbers show the expected trade-off most clearly for Yoda: the adapter fits the Yoda corpus best, but generalizes worst back to Shakespeare and general prose.

## Most surprising observation (one sentence)
The surprising part was that all three one-epoch adapters picked up surface style quickly, but Yoda transferred most cleanly while Rick and Hermione still drifted into repetitive or generic dialogue.



## 12 · Submission checklist

**MY ANSWER**

## Final answers from the saved full run
- Q1: LoRA wrapped `36` GPT-2 projections and trained `811,008` parameters, printed as `0.811M / 125.3M = 0.65%`. Per wrapped linear, LoRA adds `r * (d_in + d_out)` parameters.
- Q2: Fine-tuning moved the samples from generic modern prose into play-like Shakespeare text. The saved outputs add speaker labels such as `PAUL:`, `LORDY:`, `ROME:`, and `ROMEO:`, line breaks, and theatrical diction such as `O my lord`, `I beg thee`, `honour`, `bride`, and `roses of heaven`.
- Q3: Shakespeare-val PPL improved from `93.53` to `41.58` (`-51.94`). Control P&P PPL worsened mildly from `16.92` to `18.62` (`+1.71`). The trade-off is strong Shakespeare specialization with a small general-English cost.
- Q4: Hand-rolled LoRA and PEFT both trained `811,008` parameters over the same `36` targeted projections. PEFT printed `811,008 / 125,250,816 = 0.6475%`; the hand-rolled path printed about `0.65%`.
- Q5: PEFT reached a very similar validation result: hand-rolled Shakespeare-val PPL `41.58` vs PEFT `41.45`, and hand-rolled control PPL `18.62` vs PEFT `18.85`. Small differences can come from initialization, dataloader order, dropout samples, and PEFT's direct `Conv1D` handling.
- Q6: The clean bonus runs used Rick, Hermione, and Yoda, not three Yoda samples. Exact PPLs were Rick style/Shakespeare/control `26.08 / 77.26 / 17.01`, Hermione `20.57 / 90.89 / 18.53`, and Yoda `11.61 / 123.61 / 26.70`. Compared with the untuned baseline `93.53 / 16.92`, Rick and Hermione still improved Shakespeare PPL slightly but did not approach the Shakespeare adapter; Yoda worsened both Shakespeare and control PPL.

## Checklist status
- [ X ] All cells run top to bottom without error in the saved notebook outputs.
- [ X ] Q1-Q5 answered; Q6 answered from the clean Rick/Hermione/Yoda bonus outputs and exact PPL cell.
- [ X ] Baseline and post-fine-tune sample outputs are visible in the saved notebook.
- [ X ] All four main PPL numbers are printed, plus exact Q6 style/Shakespeare/control PPLs for Rick, Hermione, and Yoda.
- [ X ] `lora_adapter.pt` round-trip assertion passes with `max abs diff: 0.0`.
- [ X ] Hand-rolled vs PEFT comparison runs and PPLs are reported side-by-side.
